# Clase 8 — Visualización avanzada y dashboards
**Taller de Ciencia de Datos (EIN146A25) · 2026-2 · Jueves 1 de octubre**

Cuatro ejercicios sobre un mini-caso de **asistencia escolar en las 12 comunas de la provincia de Concepción**. La Clase 7 enseñó a mirar los datos; hoy se sube un nivel: **cuánta certeza** hay detrás de una diferencia, cómo leer el **tiempo**, cómo comparar **muchos grupos y el territorio**, y cómo armar un **dashboard** mínimo para explorar. Al final, la sección que se agrega a `03-eda.ipynb` de su proyecto.

**Datos sintéticos generados para el lab: ninguna cifra describe a una comuna, establecimiento o estudiante real.**

| # | Ejercicio | Qué se practica | Min |
|---|---|---|---|
| 1 | Incertidumbre: intervalos y embudo | intervalo de Wilson, barras de error, gráfico de embudo | 20 |
| 2 | Tiempo: estacionalidad y tendencia | `pd.to_datetime`, serie mensual, mes calendario, una línea por año | 20 |
| 3 | Grupos y territorio: paneles y cartograma | `subplots(sharey=True)`, residuos para anomalías, cartograma de cuadrícula | 20 |
| 4 | Dashboard mínimo | función `panel_comuna()` en Matplotlib; (opcional) HTML de Plotly con menú | 30 |

Reglas del notebook:
- Los archivos de práctica **se generan solos** en `data_practica/`: no hay que descargar nada y el lab no usa internet.
- Las celdas marcadas **✏️ TU TURNO** son las que deben completar.
- Cada ejercicio trae su **verificación automática** (debe imprimir todos los ✓) y una **solución propuesta** plegada: intenten primero sin mirarla.
- Cada ejercicio lee los CSV por su cuenta; si uno no les sale, el siguiente puede usar las variables de referencia `_xxx` que deja la celda del profesor.
- `seaborn` y `plotly` son **opcionales**: los Ejercicios 1 a 3 y la parte A del 4 funcionan solo con Matplotlib y pandas.
- Ver `lab08-guia-visualizacion.md` para la referencia técnica de cada sección (§N).

In [ ]:
# Setup — ejecutar primero
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

try:
    import seaborn as sns; HAY_SNS = True
except ImportError:
    HAY_SNS = False
try:
    import plotly.graph_objects as go; HAY_PLOTLY = True
except ImportError:
    HAY_PLOTLY = False
print(f"seaborn: {'sí' if HAY_SNS else 'no'} · plotly: {'sí' if HAY_PLOTLY else 'no'}")

CARPETA = Path("data_practica")
CARPETA.mkdir(exist_ok=True)
FIGS = Path("figs")

MESES = {3: "mar", 4: "abr", 5: "may", 6: "jun", 7: "jul", 8: "ago", 9: "sep", 10: "oct", 11: "nov", 12: "dic"}

# Cartograma de cuadrícula: (columna, fila) de cada comuna; fila 0 = norte. Aproximado, no a escala.
GRILLA = {"Tomé": (0, 0), "Penco": (0, 1), "Florida": (2, 1),
          "Talcahuano": (0, 2), "Concepción": (1, 2), "Chiguayante": (2, 2), "Hualqui": (3, 2),
          "Hualpén": (0, 3), "San Pedro de la Paz": (1, 3), "Santa Juana": (3, 3),
          "Coronel": (1, 4), "Lota": (1, 5)}


def guardar_figura(fig, nombre):
    '''Guarda una figura en figs/ (la crea si no existe) con la resolución del informe.'''
    FIGS.mkdir(exist_ok=True)
    ruta = FIGS / nombre
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    print("Guardado:", ruta)


def ejecutar_checks(checks):
    '''checks: lista de (nombre, funcion_booleana_sin_argumentos).
    Atrapa NameError para que "Restart & Run All" con TU TURNO vacíos no se caiga.'''
    faltan = 0
    for nombre, chk in checks:
        try:
            ok = bool(chk())
        except NameError as e:
            var = str(e).split("'")[1]
            print(f"✗ {nombre} (todavía no definen `{var}`)")
            faltan += 1
            continue
        except Exception as e:
            print(f"✗ {nombre} (error al evaluar: {e})")
            faltan += 1
            continue
        print(("✓" if ok else "✗"), nombre)
        faltan += not ok
    print("Todo OK" if faltan == 0 else f"Faltan {faltan}")
    return faltan == 0


print("Python:", sys.version.split()[0], "· pandas:", pd.__version__)
print("Carpeta de trabajo:", CARPETA.resolve())

In [ ]:
# Genera los archivos de práctica (no modificar esta celda)
rng = np.random.default_rng(2026)
COMUNAS = [("08101", "Concepción"), ("08102", "Coronel"), ("08103", "Chiguayante"), ("08104", "Florida"),
           ("08105", "Hualqui"), ("08106", "Lota"), ("08107", "Penco"), ("08108", "San Pedro de la Paz"),
           ("08109", "Santa Juana"), ("08110", "Talcahuano"), ("08111", "Tomé"), ("08112", "Hualpén")]

# --- Archivo 1: comunas_riesgo.csv (conteos fijos) ---
RIESGO = {"Concepción": (5200, 598), "Talcahuano": (4300, 533), "San Pedro de la Paz": (3700, 420),
          "Coronel": (3300, 396), "Hualpén": (2400, 312), "Chiguayante": (2200, 250), "Penco": (1600, 190),
          "Tomé": (1500, 210), "Lota": (1100, 165), "Hualqui": (520, 63), "Florida": (150, 25),
          "Santa Juana": (120, 8)}
comunas_riesgo = pd.DataFrame([(c, n, *RIESGO[n]) for c, n in COMUNAS],
                              columns=["cod_comuna", "comuna", "estudiantes", "en_riesgo"])

# --- Archivo 2: asistencia_mensual.csv ---
DEPS = ["Pública", "Part. subvencionado", "Part. pagado"]
BASE = {"Pública": 86.5, "Part. subvencionado": 90.0, "Part. pagado": 93.5}
MAT = {"Pública": (350, 900), "Part. subvencionado": (250, 700), "Part. pagado": (150, 400)}
TEND = {"Pública": -1.0, "Part. subvencionado": 0.0, "Part. pagado": 0.0}   # pp por año
EXTRA = {"Coronel": -1.5, "Lota": -2.0}                                     # pp por año, solo Pública
EST = {3: 1.0, 4: 1.5, 5: 0.5, 6: -3.5, 7: -5.0, 8: 0.0, 9: -2.5, 10: 0.5, 11: 1.0, 12: -3.0}

est, usados = [], set()
for cod, nom in COMUNAS:
    for d in DEPS:
        for _ in range(int(rng.integers(2, 7))):
            while True:
                r = int(rng.integers(40000, 50000))
                if r not in usados:
                    usados.add(r); break
            lo, hi = MAT[d]
            est.append((str(r), cod, nom, d, int(rng.integers(lo, hi + 1)), float(rng.normal(0, 1.5))))
E = pd.DataFrame(est, columns=["rbd", "cod_comuna", "comuna", "dependencia", "matricula", "ef"])

recs = []
for _, e in E.iterrows():
    for yi, anio in enumerate([2023, 2024, 2025]):
        for mes in range(3, 13):
            a = BASE[e.dependencia] + EST[mes] + e.ef + rng.normal(0, 0.8)
            if e.dependencia == "Pública":
                a += (TEND["Pública"] + EXTRA.get(e.comuna, 0)) * yi
            if e.comuna == "Lota" and e.dependencia == "Pública" and anio == 2024 and mes == 10:
                a -= 22
            recs.append((e.rbd, e.cod_comuna, e.comuna, e.dependencia, f"{anio}-{mes:02d}-01",
                         anio, mes, e.matricula, round(float(np.clip(a, 40, 100)), 1)))
A = pd.DataFrame(recs, columns=["rbd", "cod_comuna", "comuna", "dependencia", "fecha",
                                "anio", "mes", "matricula", "asistencia_pct"])
elig = A.index[~((A.comuna == "Lota") & (A.dependencia == "Pública") & (A.anio == 2024) & (A.mes == 10))]
A.loc[rng.choice(elig, 40, replace=False), "asistencia_pct"] = 0.0     # nulo disfrazado
A = A.sample(frac=1, random_state=2026).reset_index(drop=True)

# rbd y cod_comuna son texto (con cero a la izquierda): al leer, usar dtype=str
comunas_riesgo.to_csv(CARPETA / "comunas_riesgo.csv", index=False, encoding="utf-8")
A.to_csv(CARPETA / "asistencia_mensual.csv", index=False, encoding="utf-8")
print("comunas_riesgo.csv:", comunas_riesgo.shape)
print("asistencia_mensual.csv:", A.shape)
print("Escritos en:", CARPETA.resolve())

---
## Ejercicio 1 — Incertidumbre: intervalos y embudo (≈20 min)

`comunas_riesgo.csv` trae, para cada comuna, cuántos estudiantes hay y cuántos están en riesgo de deserción. La pregunta del municipio es simple: **¿dónde intervenir primero?** El problema es que una tasa calculada sobre 120 estudiantes no tiene la misma certeza que una calculada sobre 5.200.

Ver `lab08-guia-visualizacion.md`, §1–3.

### 1.1 Primero, el error
El ranking de tasas en barras horizontales, **sin n ni intervalo**.

In [ ]:
cr = pd.read_csv(CARPETA / "comunas_riesgo.csv", dtype={"cod_comuna": str})
cr["tasa"] = cr.en_riesgo / cr.estudiantes
orden = cr.sort_values("tasa")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(orden.comuna, orden.tasa * 100)
ax.set_xlabel("estudiantes en riesgo (%)")
ax.set_title("Ranking de riesgo por comuna")

*¿Qué comuna intervendrían primero con este gráfico? ¿Y cuál dejarían para el final?*

### 1.2 ✏️ TU TURNO
Construyan:

1. `df1` = la tabla de `comunas_riesgo.csv` con columnas nuevas `tasa`, `lo`, `hi`: la tasa y su **intervalo de Wilson al 95 %**, en proporción (0 a 1, no en %). Con $\hat p = k/n$ y $z = 1{,}96$:
   $$\text{centro} = \frac{\hat p + \frac{z^2}{2n}}{1 + \frac{z^2}{n}} \qquad \text{margen} = \frac{z}{1 + \frac{z^2}{n}}\sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}}$$
   `lo = centro − margen`, `hi = centro + margen` (guía §2).
2. `p_global` = `en_riesgo.sum() / estudiantes.sum()`.
3. `ax1` = puntos horizontales con barras de error (`errorbar`), ordenados por tasa, con una línea vertical en `p_global`; título = hallazgo y ejes con unidades.
4. `ax1b` = **gráfico de embudo**: tasa vs. `estudiantes` con el eje X **logarítmico**, la línea horizontal de `p_global` y las dos curvas de límite al 95 %: `p_global ± 1,96·√(p_global(1−p_global)/n)` (guía §3).
5. `fuera` = conjunto (`set`) con los nombres de las comunas que quedan fuera del embudo.
6. `conclusion1` = `"A"`, `"B"` o `"C"`:
   - A) Florida tiene la tasa más alta: hay que intervenir Florida primero.
   - B) Florida y Santa Juana tienen las tasas más extremas, pero con n = 150 y 120 sus intervalos incluyen la tasa global; Lota y Tomé, con una diferencia puntual menor, son las que se distinguen de ella.
   - C) Con intervalos de confianza no se puede concluir nada de ninguna comuna.

Pistas: `wilson()` recibe columnas completas (`df1.en_riesgo`, `df1.estudiantes`) y devuelve dos columnas · `errorbar` pide `xerr=[tasa − lo, hi − tasa]` (distancias, no límites) · para el embudo, evaluar los límites sobre `n = np.logspace(2, 4, 200)`.

In [ ]:
# ✏️ TU TURNO — completen (descomenten y reemplacen los ...)
# def wilson(k, n, z=1.96):
#     p = k / n
#     centro = ...
#     margen = ...
#     return centro - margen, centro + margen
#
# df1 = pd.read_csv(CARPETA / "comunas_riesgo.csv", dtype={"cod_comuna": str})
# df1["tasa"] = ...
# df1["lo"], df1["hi"] = wilson(df1.en_riesgo, df1.estudiantes)
# p_global = ...
#
# d = df1.sort_values("tasa")
# fig1, ax1 = plt.subplots(figsize=(7, 4.5))
# ax1.errorbar(d.tasa * 100, np.arange(len(d)), xerr=[..., ...], fmt="o", capsize=3)
# ax1.set_yticks(np.arange(len(d)), d.comuna)
# ax1.axvline(...)
# ax1.set_xlabel(...); ax1.set_title(...)
#
# fig1b, ax1b = plt.subplots(figsize=(7, 4.5))
# n = np.logspace(2, 4, 200)
# ax1b.scatter(...)
# ax1b.axhline(...)
# ax1b.plot(n, ..., "k--"); ax1b.plot(n, ..., "k--")
# ax1b.set_xscale("log")
#
# fuera = set(...)
# conclusion1 = ...

### ✅ Verificación automática
Ejecuten esta celda cuando tengan `df1`, `p_global`, `ax1`, `ax1b`, `fuera` y `conclusion1`. Debe imprimir 7 ✓.

In [ ]:
from matplotlib.container import ErrorbarContainer


def _wilson(k, n, z=1.96):
    p = k / n
    centro = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    margen = z / (1 + z**2 / n) * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return centro - margen, centro + margen


_ref1 = pd.read_csv(CARPETA / "comunas_riesgo.csv", dtype={"cod_comuna": str}).set_index("comuna")
_ref1["lo"], _ref1["hi"] = _wilson(_ref1.en_riesgo, _ref1.estudiantes)


def _n_puntos(ax):
    n = [len(c.get_offsets()) for c in ax.collections]
    n += [len(l.get_xdata()) for l in ax.lines if l.get_linestyle() in ("None", "", " ")]
    return max(n, default=0)


def verificar_ej1():
    checks = [
        ("df1 tiene 12 filas y las columnas tasa, lo, hi",
         lambda: len(df1) == 12 and {"tasa", "lo", "hi"} <= set(df1.columns)),
        ("p_global ≈ 0,1215 (±0,0005)",
         lambda: abs(p_global - 0.1215) < 0.0005),
        ("lo y hi coinciden con la referencia (±0,001, en proporción)",
         lambda: np.allclose(df1.set_index("comuna").loc[_ref1.index, ["lo", "hi"]],
                             _ref1[["lo", "hi"]], atol=0.001)),
        ("ax1 tiene barras de error para las 12 comunas",
         lambda: any(isinstance(c, ErrorbarContainer) and len(c.lines[2][0].get_segments()) == 12
                     for c in ax1.containers)),
        ("ax1b: eje X logarítmico, 12 puntos y al menos 3 líneas",
         lambda: ax1b.get_xscale() == "log" and _n_puntos(ax1b) == 12
                 and sum(l.get_linestyle() not in ("None", "", " ") for l in ax1b.lines) >= 3),
        ('fuera == {"Lota", "Tomé"}',
         lambda: set(fuera) == {"Lota", "Tomé"}),
        ('conclusion1 == "B"',
         lambda: conclusion1 == "B"),
    ]
    return ejecutar_checks(checks)

verificar_ej1()

<details>
<summary><b>Solución propuesta (Ejercicio 1)</b> — clic para desplegar</summary>

```python
def wilson(k, n, z=1.96):
    p = k / n
    centro = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    margen = z / (1 + z**2 / n) * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return centro - margen, centro + margen

df1 = pd.read_csv(CARPETA / "comunas_riesgo.csv", dtype={"cod_comuna": str})
df1["tasa"] = df1.en_riesgo / df1.estudiantes
df1["lo"], df1["hi"] = wilson(df1.en_riesgo, df1.estudiantes)
p_global = df1.en_riesgo.sum() / df1.estudiantes.sum()

# Puntos con intervalo, ordenados por tasa
d = df1.sort_values("tasa")
y = np.arange(len(d))
fig1, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.errorbar(d.tasa * 100, y, xerr=[(d.tasa - d.lo) * 100, (d.hi - d.tasa) * 100], fmt="o", capsize=3)
ax1.set_yticks(y, [f"{c} (n={n})" for c, n in zip(d.comuna, d.estudiantes)])
ax1.axvline(p_global * 100, color="gray", linestyle="--", label=f"tasa global ({p_global:.1%})")
ax1.set_xlabel("estudiantes en riesgo (%) · intervalo de Wilson 95 %")
ax1.set_title("Los extremos del ranking son las comunas con menos estudiantes")
ax1.legend(loc="lower right")

# Embudo
fig1b, ax1b = plt.subplots(figsize=(7, 4.5))
n = np.logspace(2, 4, 200)
se = np.sqrt(p_global * (1 - p_global) / n)
ax1b.scatter(df1.estudiantes, df1.tasa * 100, zorder=3)
for _, r in df1.iterrows():
    ax1b.annotate(r.comuna, (r.estudiantes, r.tasa * 100), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax1b.axhline(p_global * 100, color="gray")
ax1b.plot(n, (p_global + 1.96 * se) * 100, "k--", lw=1, label="límites 95 %")
ax1b.plot(n, (p_global - 1.96 * se) * 100, "k--", lw=1)
ax1b.set_xscale("log")
ax1b.set_xlabel("estudiantes de la comuna (escala log)"); ax1b.set_ylabel("en riesgo (%)")
ax1b.set_title("Solo Lota y Tomé quedan fuera del embudo del 95 %")
ax1b.legend()

se_comuna = np.sqrt(p_global * (1 - p_global) / df1.estudiantes)
fuera = set(df1.loc[(df1.tasa - p_global).abs() > 1.96 * se_comuna, "comuna"])
conclusion1 = "B"
verificar_ej1()
```

Florida (16,7 %) y Santa Juana (6,7 %) encabezan y cierran el ranking, pero con 150 y 120 estudiantes sus intervalos (≈ 11,5–23,5 % y ≈ 3,4–12,6 %) incluyen la tasa global de 12,15 %: con tan pocos estudiantes, 5 casos más o menos mueven la tasa varios puntos. Lota (15,0 %, n = 1.100) y Tomé (14,0 %, n = 1.500) tienen una diferencia puntual menor, pero con n grande quedan **fuera del embudo, por arriba**: son las que se distinguen de la provincia. Con 12 comunas y límites al 95 % se esperan ~0,6 comunas fuera por azar, así que dos fuera, y del mismo lado, sí es una señal para mirar.

</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
_df1 = _ref1.reset_index()[["cod_comuna", "comuna", "estudiantes", "en_riesgo", "lo", "hi"]]
_df1.insert(4, "tasa", _df1.en_riesgo / _df1.estudiantes)
_p_global = _df1.en_riesgo.sum() / _df1.estudiantes.sum()
_se = np.sqrt(_p_global * (1 - _p_global) / _df1.estudiantes)
_fuera = set(_df1.loc[(_df1.tasa - _p_global).abs() > 1.96 * _se, "comuna"])

assert (_df1.estudiantes.sum(), _df1.en_riesgo.sum()) == (26090, 3170)
assert abs(_p_global - 0.1215) < 0.0005
assert _fuera == {"Lota", "Tomé"}, "La solución de referencia del Ej. 1 no pasa la verificación"
print(f"Estudiantes: {_df1.estudiantes.sum()} · en riesgo: {_df1.en_riesgo.sum()} · tasa global: {_p_global:.2%}")
print("Fuera del embudo:", sorted(_fuera))

---
## Ejercicio 2 — Tiempo: estacionalidad y tendencia (≈20 min)

`asistencia_mensual.csv` trae la asistencia de 146 establecimientos, de marzo a diciembre de 2023, 2024 y 2025 (30 meses). Dos preguntas que suelen mezclarse: ¿la asistencia **viene cayendo** (tendencia) o solo **baja en ciertos meses** (estacionalidad)?

Ver `lab08-guia-visualizacion.md`, §4–5.

### 2.1 Primero, el error
Promedio mensual de `asistencia_pct` **con los datos tal como vienen** y una sola línea para los 30 meses.

In [ ]:
df = pd.read_csv(CARPETA / "asistencia_mensual.csv", dtype={"rbd": str, "cod_comuna": str})
serie_cruda = df.groupby("fecha").asistencia_pct.mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(serie_cruda.index, serie_cruda.values, marker="o")
ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.set_ylabel("asistencia (%)")
ax.set_title("Asistencia promedio mensual (datos crudos)")

*Caídas irregulares y una sospecha: ¿la asistencia viene cayendo? ¿Por qué baja en julio? ¿Todas esas caídas son reales?*

### 2.2 ✏️ TU TURNO
A partir de `df`:

1. `n_ceros` = cantidad de filas con `asistencia_pct == 0.0` (un establecimiento que no reportó, no uno vacío). `df2` = copia de `df` con esos valores convertidos a `NaN` y `fecha` convertida con `pd.to_datetime`. Mismas filas y columnas; el CSV no se toca.
2. `serie` = promedio mensual de `asistencia_pct` por `fecha` (30 puntos, índice de fechas).
3. `estac` = promedio de `serie` por **mes calendario** (índice 3 a 12).
4. `yoy` = tabla mes (filas) × año (columnas) con el promedio de `asistencia_pct` (10 × 3).
5. `ax2` = **una línea por año** sobre el eje de meses calendario, con etiquetas de mes (`MESES` del Setup) y leyenda; título = hallazgo.
6. `conclusion2` = `"A"`, `"B"` o `"C"`:
   - A) La caída de julio es una tendencia: la asistencia viene cayendo mes a mes desde marzo.
   - B) La caída de julio se repite en los tres años (estacionalidad); la tendencia de fondo es una baja suave de ~1 punto en dos años, que se ve comparando el mismo mes entre años.
   - C) Los 40 ceros explican la caída de julio: al sacarlos desaparece.

Pistas: `serie.groupby(serie.index.month).mean()` · `df2.pivot_table(index="mes", columns="anio", values="asistencia_pct", aggfunc="mean")` · `ax2.set_xticks(list(MESES), list(MESES.values()))`.

In [ ]:
# ✏️ TU TURNO — completen (descomenten y reemplacen los ...)
# n_ceros = ...
# df2 = df.copy()
# df2.loc[..., "asistencia_pct"] = np.nan
# df2["fecha"] = pd.to_datetime(...)
#
# serie = df2.groupby(...).asistencia_pct.mean()
# estac = serie.groupby(serie.index.month).mean()
# yoy = df2.pivot_table(index=..., columns=..., values="asistencia_pct", aggfunc="mean")
#
# fig2, ax2 = plt.subplots(figsize=(8, 4.5))
# for anio in yoy.columns:
#     ax2.plot(...)
# ax2.set_xticks(list(MESES), list(MESES.values()))
# ax2.set_xlabel(...); ax2.set_ylabel(...); ax2.set_title(...); ax2.legend()
#
# conclusion2 = ...

### ✅ Verificación automática
Debe imprimir 8 ✓.

In [ ]:
def verificar_ej2():
    checks = [
        ("n_ceros == 40",
         lambda: n_ceros == 40),
        ("df2 tiene 4.380 filas y 9 columnas, 40 NaN y ningún 0,0 en asistencia_pct",
         lambda: df2.shape == (4380, 9) and int(df2.asistencia_pct.isna().sum()) == 40
                 and int((df2.asistencia_pct == 0.0).sum()) == 0),
        ("df2.fecha es de tipo fecha",
         lambda: pd.api.types.is_datetime64_any_dtype(df2.fecha)),
        ("serie tiene 30 puntos con índice de fechas",
         lambda: len(serie) == 30 and isinstance(serie.index, pd.DatetimeIndex)),
        ("estac: el mes más bajo es julio (7) y agosto supera a julio por más de 4 pp",
         lambda: estac.idxmin() == 7 and estac[8] - estac[7] > 4),
        ("yoy es 10 × 3 y 2025 queda bajo 2023",
         lambda: yoy.shape == (10, 3) and yoy[2025].mean() < yoy[2023].mean()),
        ("ax2 tiene 3 líneas (una por año)",
         lambda: len(ax2.lines) == 3),
        ('conclusion2 == "B"',
         lambda: conclusion2 == "B"),
    ]
    return ejecutar_checks(checks)

verificar_ej2()

<details>
<summary><b>Solución propuesta (Ejercicio 2)</b> — clic para desplegar</summary>

```python
n_ceros = int((df.asistencia_pct == 0.0).sum())
df2 = df.copy()
df2.loc[df2.asistencia_pct == 0.0, "asistencia_pct"] = np.nan
df2["fecha"] = pd.to_datetime(df2.fecha)

serie = df2.groupby("fecha").asistencia_pct.mean()
estac = serie.groupby(serie.index.month).mean()
yoy = df2.pivot_table(index="mes", columns="anio", values="asistencia_pct", aggfunc="mean")

fig2, ax2 = plt.subplots(figsize=(8, 4.5))
for anio in yoy.columns:
    ax2.plot(yoy.index, yoy[anio], marker="o", label=str(anio))
ax2.set_xticks(list(MESES), list(MESES.values()))
ax2.set_xlabel("mes"); ax2.set_ylabel("asistencia promedio (%)")
ax2.set_title("Julio es el mes más bajo los tres años; cada año la curva baja un poco")
ax2.legend(title="año")
# variante seaborn: sns.lineplot(df2, x="mes", y="asistencia_pct", hue="anio", errorbar=None, ax=ax2)

conclusion2 = "B"
verificar_ej2()
```

Los 40 ceros son establecimientos **sin reporte**: con ellos adentro, el promedio del mes baja hasta ~2,3 puntos de forma aleatoria (media ≈ 0,8), y esas caídas no significan nada. Sin ellos, el patrón se ordena: la asistencia cae en junio, **toca fondo en julio (≈ 84,3 %)** —invierno y vacaciones— y se recupera en agosto (≈ 89,3 %). Esa curva se repite en los tres años: es **estacionalidad**, no tendencia. La tendencia está en la separación entre las líneas: el promedio anual pasa de ≈ 88,9 % (2023) a ≈ 88,3 % (2024) y ≈ 87,9 % (2025). Para verla hay que comparar el mismo mes entre años, no julio contra marzo.

</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
_df2 = pd.read_csv(CARPETA / "asistencia_mensual.csv", dtype={"rbd": str, "cod_comuna": str})
_df2.loc[_df2.asistencia_pct == 0.0, "asistencia_pct"] = np.nan
_df2["fecha"] = pd.to_datetime(_df2.fecha)
_serie = _df2.groupby("fecha").asistencia_pct.mean()
_estac = _serie.groupby(_serie.index.month).mean()
_yoy = _df2.pivot_table(index="mes", columns="anio", values="asistencia_pct", aggfunc="mean")

assert int(_df2.asistencia_pct.isna().sum()) == 40
assert _estac.idxmin() == 7 and _estac[8] - _estac[7] > 4
assert _yoy[2023].mean() > _yoy[2024].mean() > _yoy[2025].mean(), "La solución de referencia del Ej. 2 no pasa la verificación"
print("Mes más bajo:", MESES[_estac.idxmin()], f"({_estac.min():.1f} %)")
print("Promedio anual:", _df2.groupby("anio").asistencia_pct.mean().round(1).to_dict())

---
## Ejercicio 3 — Grupos y territorio: paneles y cartograma (≈20 min)

La serie provincial esconde lo que pasa en cada comuna y en cada tipo de establecimiento. Con 12 comunas × 3 dependencias hay 36 series: ¿cómo se miran todas sin que el gráfico se vuelva ilegible? ¿Y cómo se ve si el problema tiene un patrón en el **territorio**?

Ver `lab08-guia-visualizacion.md`, §5–7.

### 3.1 Primero, el error
Las 12 comunas en **un solo gráfico**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
_df2.groupby(["fecha", "comuna"]).asistencia_pct.mean().unstack().plot(ax=ax)
ax.set_ylabel("asistencia (%)")
ax.set_title("Asistencia mensual de las 12 comunas")

*¿Qué comuna se comporta distinto?*

### 3.2 ✏️ TU TURNO
Sobre `_df2` (la referencia del Ej. 2) o su propio `df2`:

1. `fig3, axs3` = cuadrícula de **3 × 4 paneles** con `sharex=True, sharey=True`: una comuna por panel (en orden `sorted`), **3 líneas por panel (una por dependencia)** y título = nombre de la comuna.
2. `anomalia3` = tupla `(comuna, dependencia, año, mes)` con la mayor caída respecto de lo normal. Residuo = promedio por comuna × dependencia × año × mes, **menos** el promedio de ese mismo mes calendario en esa comuna y dependencia (los tres años). La anomalía es el residuo mínimo.
3. `cambio` = `Series` (índice = comuna) con el cambio de la asistencia promedio **2025 vs. 2023** en establecimientos `Pública`, en puntos porcentuales (pp).
4. `ax3b` = **cartograma de cuadrícula** con `GRILLA` (Setup): un `Rectangle` por comuna, coloreado por `cambio` (un solo mapa de color, con barra de color), y una etiqueta con el nombre y el valor. Eje Y invertido (norte arriba), sin ejes visibles.
5. `conclusion3` = `"A"`, `"B"` o `"C"`:
   - A) El deterioro de asistencia en establecimientos públicos es parejo en toda la provincia.
   - B) Todas las comunas caen ~2 pp, pero Lota y Coronel, vecinas en el sur, caen 5–6 pp: el deterioro se concentra ahí.
   - C) Lota cae más solo por la anomalía de octubre de 2024; sin ella no hay diferencia.

Pistas: `g = df.groupby(["comuna", "dependencia", "anio", "mes"]).asistencia_pct.mean()` y `g.groupby(level=["comuna", "dependencia", "mes"]).transform("mean")` · `pivot_table` o `unstack()` por año para `cambio` · `cmap = plt.get_cmap("Reds_r")`, `norm = plt.Normalize(vmin, vmax)`, `color=cmap(norm(valor))` y `fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax3b)`.

In [ ]:
# ✏️ TU TURNO — completen (descomenten y reemplacen los ...)
# comunas = sorted(_df2.comuna.unique())
# fig3, axs3 = plt.subplots(3, 4, figsize=(14, 8), sharex=True, sharey=True)
# for ax, c in zip(axs3.flat, comunas):
#     s = ...                      # fecha × dependencia para la comuna c
#     for dep in s.columns:
#         ax.plot(...)
#     ax.set_title(c, fontsize=9)
#
# g = _df2.groupby(["comuna", "dependencia", "anio", "mes"]).asistencia_pct.mean()
# residuo = g - ...
# c, d, a, m = residuo.idxmin()
# anomalia3 = (c, d, int(a), int(m))
#
# cambio = ...
#
# fig3b, ax3b = plt.subplots(figsize=(5, 7))
# cmap = plt.get_cmap("Reds_r"); norm = plt.Normalize(vmin=cambio.min(), vmax=0)
# for comuna, (col, fila) in GRILLA.items():
#     ax3b.add_patch(Rectangle((col, fila), 0.95, 0.95, color=...))
#     ax3b.text(col + 0.475, fila + 0.475, ..., ha="center", va="center", fontsize=7)
# ax3b.set_xlim(0, 4); ax3b.set_ylim(6, 0); ax3b.axis("off")
# fig3b.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax3b, label=...)
#
# conclusion3 = ...

### ✅ Verificación automática
Debe imprimir 6 ✓.

In [ ]:
_pub = _df2[_df2.dependencia == "Pública"].groupby(["comuna", "anio"]).asistencia_pct.mean().unstack()
_cambio = _pub[2025] - _pub[2023]


def verificar_ej3():
    checks = [
        ("fig3 tiene 12 paneles con el mismo eje Y y al menos 3 líneas cada uno",
         lambda: len(fig3.axes) == 12 and len({ax.get_ylim() for ax in fig3.axes}) == 1
                 and all(len(ax.lines) >= 3 for ax in fig3.axes)),
        ('anomalia3 == ("Lota", "Pública", 2024, 10)',
         lambda: tuple(anomalia3) == ("Lota", "Pública", 2024, 10)),
        ("cambio tiene 12 valores y coincide con la referencia (±0,05 pp)",
         lambda: len(cambio) == 12 and (cambio - _cambio).abs().max() < 0.05),
        ('cambio.idxmin() == "Lota"',
         lambda: cambio.idxmin() == "Lota"),
        ("ax3b tiene 12 rectángulos y 12 etiquetas",
         lambda: sum(isinstance(p, Rectangle) for p in ax3b.patches) == 12 and len(ax3b.texts) >= 12),
        ('conclusion3 == "B"',
         lambda: conclusion3 == "B"),
    ]
    return ejecutar_checks(checks)

verificar_ej3()

<details>
<summary><b>Solución propuesta (Ejercicio 3)</b> — clic para desplegar</summary>

```python
import matplotlib.dates as mdates

comunas = sorted(_df2.comuna.unique())
fig3, axs3 = plt.subplots(3, 4, figsize=(14, 8), sharex=True, sharey=True)
for ax, c in zip(axs3.flat, comunas):
    s = _df2[_df2.comuna == c].groupby(["fecha", "dependencia"]).asistencia_pct.mean().unstack()
    for dep in s.columns:
        ax.plot(s.index, s[dep], lw=1, label=dep)
    ax.set_title(c, fontsize=9)
axs3[0, 0].legend(fontsize=7)
fig3.supylabel("asistencia (%)")
fig3.suptitle("Lota (pública) cae en octubre de 2024; el resto sigue el ciclo del año")
axs3[0, 0].xaxis.set_major_locator(mdates.YearLocator())          # con sharex, vale para los 12
axs3[0, 0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
# variante seaborn: sns.relplot(_df2, x="fecha", y="asistencia_pct", hue="dependencia", col="comuna",
#                               col_wrap=4, kind="line", errorbar=None, height=2.2)

g = _df2.groupby(["comuna", "dependencia", "anio", "mes"]).asistencia_pct.mean()
residuo = g - g.groupby(level=["comuna", "dependencia", "mes"]).transform("mean")
c, d, a, m = residuo.idxmin()
anomalia3 = (c, d, int(a), int(m))

pub = _df2[_df2.dependencia == "Pública"].pivot_table(index="comuna", columns="anio",
                                                      values="asistencia_pct", aggfunc="mean")
cambio = pub[2025] - pub[2023]

fig3b, ax3b = plt.subplots(figsize=(5, 7))
cmap = plt.get_cmap("Reds_r")
norm = plt.Normalize(vmin=cambio.min(), vmax=0)
for comuna, (col, fila) in GRILLA.items():
    v = cambio[comuna]
    ax3b.add_patch(Rectangle((col, fila), 0.95, 0.95, color=cmap(norm(v))))
    etiqueta = comuna.replace(" de ", "\nde ", 1)   # "San Pedro de la Paz" en dos líneas
    ax3b.text(col + 0.475, fila + 0.475, f"{etiqueta}\n{v:+.1f} pp", ha="center", va="center",
              fontsize=7, color="white" if v < -4 else "black")
ax3b.set_xlim(0, 4); ax3b.set_ylim(6, 0)          # ylim al revés: fila 0 (norte) arriba
ax3b.set_aspect("equal"); ax3b.axis("off")
fig3b.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax3b, shrink=0.6,
               label="cambio de asistencia 2025 vs. 2023 (pp)")
ax3b.set_title("Públicos: Lota y Coronel caen 5–6 pp; el resto, ~2 pp")

conclusion3 = "B"
verificar_ej3()
```

Con **paneles de ejes compartidos** cada comuna se lee sola y se compara con las demás a la misma escala: salta a la vista el pozo de Lota pública en octubre de 2024 (≈ −14 pp respecto de su octubre típico). El residuo lo encuentra sin mirar: es el mínimo único de las 1.080 celdas comuna × dependencia × mes.

El cartograma responde otra pregunta: **dónde** se concentra el deterioro. En los establecimientos públicos todas las comunas caen ~2 pp entre 2023 y 2025, pero **Lota (≈ −6,4) y Coronel (≈ −5,1)**, vecinas en el sur, caen casi el triple. La anomalía de 2024 no explica esto (el cambio compara 2025 con 2023; 2024 no entra en la cuenta): el deterioro es sostenido. Ojo: el cartograma pinta un **cambio**, no un conteo; pintar "número de estudiantes" solo mostraría qué comunas son más grandes.

</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
_g = _df2.groupby(["comuna", "dependencia", "anio", "mes"]).asistencia_pct.mean()
_res = _g - _g.groupby(level=["comuna", "dependencia", "mes"]).transform("mean")
_c, _d, _a, _m = _res.idxmin()
_anomalia = (_c, _d, int(_a), int(_m))

assert _anomalia == ("Lota", "Pública", 2024, 10)
assert _cambio.idxmin() == "Lota" and _cambio.drop(["Lota", "Coronel"]).min() > -3, \
    "La solución de referencia del Ej. 3 no pasa la verificación"
print("Anomalía:", _anomalia, f"(residuo {_res.min():.1f} pp)")
print("Cambio 2025 vs. 2023, públicos (pp):", _cambio.sort_values().round(1).to_dict())

---
## Ejercicio 4 — Dashboard mínimo (≈30 min; la parte B es opcional)

Un dashboard exploratorio es una **función de un filtro**: se elige una comuna y todas las vistas se recalculan. La parte A (todos) lo hace con Matplotlib; la parte B (solo si tienen Plotly) lo convierte en un HTML con un menú desplegable, que se abre en cualquier navegador sin servidor.

Ver `lab08-guia-visualizacion.md`, §8.

### 4.1 Primero, el error
La vista de una comuna escrita "a mano": para ver Coronel hay que copiar la celda y cambiar `"Lota"`. Doce comunas = doce copias, y cualquier arreglo hay que hacerlo doce veces.

In [ ]:
lota = _df2[_df2.comuna == "Lota"]
fig, ax = plt.subplots(figsize=(8, 3.5))
lota.groupby(["fecha", "dependencia"]).asistencia_pct.mean().unstack().plot(ax=ax)
ax.set_title("Lota")

### 4.2 ✏️ TU TURNO — Parte A (todos, solo Matplotlib)
Escriban `panel_comuna(df, comuna)`, que devuelva una `Figure` con **3 paneles**:

1. Serie mensual de la comuna **por dependencia** (3 líneas), con leyenda.
2. Promedio de la comuna por **mes calendario** (línea o barras).
3. Una **tarjeta de texto**: un `Axes` sin ejes (`ax.axis("off")`) con el nombre de la comuna y 3 cifras: asistencia promedio de 2025, cambio vs. 2023 (en **pp**) y el mes calendario más bajo.

Tiene que funcionar para cualquiera de las 12 comunas. Llámenla con Lota: `fig4a = panel_comuna(df2, "Lota")` (o `_df2` si no hicieron el Ej. 2).

Pistas: `fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [2.2, 1.3, 1]})` · `a3.text(0.05, 0.9, ..., transform=a3.transAxes)` escribe en coordenadas del panel (0 a 1).

In [ ]:
# ✏️ TU TURNO — completen (descomenten y reemplacen los ...)
# def panel_comuna(df, comuna):
#     d = df[df.comuna == comuna]
#     fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [2.2, 1.3, 1]})
#
#     serie = ...                  # fecha × dependencia
#     for dep in serie.columns:
#         a1.plot(...)
#
#     estac = ...                  # promedio por mes calendario
#     a2.plot(...)
#
#     p25 = ...; p23 = ...
#     a3.axis("off")
#     a3.text(0.05, 0.85, comuna, fontsize=16, weight="bold", transform=a3.transAxes)
#     a3.text(0.05, 0.60, ..., transform=a3.transAxes)
#     ...
#     return fig
#
# fig4a = panel_comuna(df2, "Lota")

### ✅ Verificación automática (parte A)
Debe imprimir 4 ✓. Prueben también con otras comunas: `panel_comuna(df2, "Santa Juana")`.

In [ ]:
from matplotlib.figure import Figure
from matplotlib.text import Text


def verificar_ej4a():
    checks = [
        ("panel_comuna existe y fig4a es una Figure",
         lambda: callable(panel_comuna) and isinstance(fig4a, Figure)),
        ("fig4a tiene 3 paneles (Axes)",
         lambda: len(fig4a.axes) == 3),
        ("el panel de líneas tiene 3 series (una por dependencia)",
         lambda: any(len(ax.lines) == 3 for ax in fig4a.axes)),
        ('algún texto de la figura dice "Lota" y "pp"',
         lambda: "Lota" in (t := " ".join(x.get_text() for x in fig4a.findobj(Text))) and "pp" in t),
    ]
    return ejecutar_checks(checks)

verificar_ej4a()

<details>
<summary><b>Solución propuesta (Ejercicio 4, parte A)</b> — clic para desplegar</summary>

```python
def panel_comuna(df, comuna):
    d = df[df.comuna == comuna]
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [2.2, 1.3, 1]})

    serie = d.groupby(["fecha", "dependencia"]).asistencia_pct.mean().unstack()
    for dep in serie.columns:
        a1.plot(serie.index, serie[dep], lw=1.2, label=dep)
    a1.set_ylabel("asistencia (%)"); a1.set_title("Serie mensual por dependencia"); a1.legend(fontsize=8)

    estac = d.groupby("mes").asistencia_pct.mean()
    a2.plot(estac.index, estac.values, marker="o")
    a2.set_xticks(list(MESES), list(MESES.values()), fontsize=8)
    a2.set_ylabel("asistencia (%)"); a2.set_title("Promedio por mes calendario")

    p25 = d.loc[d.anio == 2025, "asistencia_pct"].mean()
    p23 = d.loc[d.anio == 2023, "asistencia_pct"].mean()
    a3.axis("off")
    a3.text(0.05, 0.85, comuna, fontsize=16, weight="bold", transform=a3.transAxes)
    a3.text(0.05, 0.60, f"Asistencia 2025: {p25:.1f} %", fontsize=11, transform=a3.transAxes)
    a3.text(0.05, 0.42, f"Cambio vs. 2023: {p25 - p23:+.1f} pp", fontsize=11, transform=a3.transAxes)
    a3.text(0.05, 0.24, f"Mes más bajo: {MESES[estac.idxmin()]}", fontsize=11, transform=a3.transAxes)

    fig.suptitle(f"Asistencia escolar — {comuna} (2023–2025)")
    fig.tight_layout()
    return fig

fig4a = panel_comuna(df2, "Lota")
verificar_ej4a()
```

La tarjeta de Lota dice lo que el Ej. 3 mostró por partes: asistencia 2025 más baja, un cambio negativo vs. 2023 y julio como el mes más bajo. Lo importante es que **la misma función** sirve para las 12 comunas: el filtro es el argumento.

</details>

### 4.3 ✏️ TU TURNO — Parte B (opcional, solo si `HAY_PLOTLY`)
`fig_dash` = figura de Plotly con:

1. **Una traza por comuna × dependencia** (36) más **3 trazas "Todas"** (una por dependencia, promedio provincial): 39 en total.
2. Un menú desplegable (`updatemenus`) con **13 botones** (Todas + 12 comunas); cada botón muestra solo las 3 trazas de su opción.
3. Título y ejes con unidades.

Guárdenla con `fig_dash.write_html(FIGS / "dashboard_lab08.html", include_plotlyjs=True)` (autocontenido: funciona sin internet, pesa ~3–5 MB) y ábranla en el navegador.

Pistas: cada traza es `go.Scatter(x=..., y=..., name=..., visible=...)` · agreguen las trazas en el mismo orden que los botones (primero las 3 de "Todas", después 3 por comuna) · cada botón es `dict(label=..., method="update", args=[{"visible": [...]}])`, con una lista de 39 `True`/`False`.

In [ ]:
# ✏️ TU TURNO (opcional) — completen dentro del if
if HAY_PLOTLY:
    pass
    # opciones = ["Todas"] + sorted(_df2.comuna.unique())
    # fig_dash = go.Figure()
    # for k, op in enumerate(opciones):
    #     d = _df2 if op == "Todas" else _df2[_df2.comuna == op]
    #     s = ...                  # fecha × dependencia
    #     for dep in s.columns:
    #         fig_dash.add_trace(go.Scatter(x=..., y=..., name=dep, visible=(k == 0)))
    # botones = [dict(label=op, method="update", args=[{"visible": [...]}])
    #            for k, op in enumerate(opciones)]
    # fig_dash.update_layout(updatemenus=[dict(buttons=botones)], title=..., xaxis_title=..., yaxis_title=...)
    # FIGS.mkdir(exist_ok=True)
    # fig_dash.write_html(FIGS / "dashboard_lab08.html", include_plotlyjs=True)

### ✅ Verificación automática (parte B)
Debe imprimir 3 ✓, o `⊘` si Plotly no está instalado (no cuenta como falla).

In [ ]:
def verificar_ej4b():
    if not HAY_PLOTLY:
        print("⊘ Plotly no está instalado: se omite (la parte A es el dashboard)")
        return True
    ruta = FIGS / "dashboard_lab08.html"
    checks = [
        ("fig_dash tiene 39 trazas (36 comuna × dependencia + 3 Todas)",
         lambda: len(fig_dash.data) == 39),
        ("el menú tiene 13 botones (Todas + 12 comunas)",
         lambda: len(fig_dash.layout.updatemenus[0].buttons) == 13),
        ("existe figs/dashboard_lab08.html, pesa más de 100 KB y contiene plotly",
         lambda: ruta.exists() and ruta.stat().st_size > 100_000
                 and "plotly" in ruta.read_text(encoding="utf-8")),
    ]
    return ejecutar_checks(checks)

verificar_ej4b()

<details>
<summary><b>Solución propuesta (Ejercicio 4, parte B)</b> — clic para desplegar</summary>

```python
if HAY_PLOTLY:
    COLOR = {"Pública": "#1f77b4", "Part. subvencionado": "#ff7f0e", "Part. pagado": "#2ca02c"}
    opciones = ["Todas"] + sorted(_df2.comuna.unique())
    fig_dash = go.Figure()
    for k, op in enumerate(opciones):
        d = _df2 if op == "Todas" else _df2[_df2.comuna == op]
        s = d.groupby(["fecha", "dependencia"]).asistencia_pct.mean().unstack()
        for dep in COLOR:
            fig_dash.add_trace(go.Scatter(x=s.index, y=s[dep], name=dep, mode="lines",
                                          line=dict(color=COLOR[dep]), visible=(k == 0)))
    botones = [dict(label=op, method="update",
                    args=[{"visible": [i // 3 == k for i in range(3 * len(opciones))]},
                          {"title.text": f"Asistencia mensual por dependencia — {op}"}])
               for k, op in enumerate(opciones)]
    fig_dash.update_layout(updatemenus=[dict(buttons=botones, x=1.0, y=1.18, xanchor="right")],
                           title="Asistencia mensual por dependencia — Todas",
                           xaxis_title="mes", yaxis_title="asistencia promedio (%)")
    FIGS.mkdir(exist_ok=True)
    fig_dash.write_html(FIGS / "dashboard_lab08.html", include_plotlyjs=True)
    print("Guardado:", FIGS / "dashboard_lab08.html")
    # fig_dash.show()  # para verlo dentro del notebook
verificar_ej4b()
```

Todas las trazas existen desde el principio; el menú solo cambia cuáles están visibles. Por eso el orden importa: la traza `i` pertenece a la opción `i // 3`. El HTML queda **autocontenido** (`include_plotlyjs=True`): se puede abrir sin internet ni servidor, adjuntar o subir al repo en `figs/`. Es una herramienta de **exploración**: no reemplaza a la bitácora ni a las figuras del informe.

</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
def _panel_comuna(df, comuna):
    d = df[df.comuna == comuna]
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4))
    s = d.groupby(["fecha", "dependencia"]).asistencia_pct.mean().unstack()
    for dep in s.columns:
        a1.plot(s.index, s[dep], label=dep)
    estac = d.groupby("mes").asistencia_pct.mean()
    a2.plot(estac.index, estac.values, marker="o")
    p25 = d.loc[d.anio == 2025, "asistencia_pct"].mean()
    p23 = d.loc[d.anio == 2023, "asistencia_pct"].mean()
    a3.axis("off")
    a3.text(0.05, 0.5, f"{comuna}\n2025: {p25:.1f} %\nvs. 2023: {p25 - p23:+.1f} pp\n"
                       f"mes más bajo: {MESES[estac.idxmin()]}", transform=a3.transAxes)
    return fig


for _c in GRILLA:                       # debe funcionar para las 12 comunas
    _f = _panel_comuna(_df2, _c)
    assert len(_f.axes) == 3 and len(_f.axes[0].lines) == 3, _c
    plt.close(_f)

if HAY_PLOTLY:
    _ops = ["Todas"] + sorted(_df2.comuna.unique())
    _fig_dash = go.Figure()
    for _op in _ops:
        _d = _df2 if _op == "Todas" else _df2[_df2.comuna == _op]
        _s = _d.groupby(["fecha", "dependencia"]).asistencia_pct.mean().unstack()
        for _dep in _s.columns:
            _fig_dash.add_trace(go.Scatter(x=_s.index, y=_s[_dep], name=_dep))
    _fig_dash.update_layout(updatemenus=[dict(buttons=[
        dict(label=_op, method="update", args=[{"visible": [i // 3 == k for i in range(39)]}])
        for k, _op in enumerate(_ops)])])
    assert len(_fig_dash.data) == 39 and len(_fig_dash.layout.updatemenus[0].buttons) == 13
print("Referencia del Ej. 4 OK", "(parte B incluida)" if HAY_PLOTLY else "(parte B omitida: sin Plotly)")

---
## 5 · Ahora con SU proyecto — ampliar `03-eda.ipynb` (11:35–12:05)

Lo que sigue no es un ejercicio: es la **sección nueva** que agregan al final de `notebooks/03-eda.ipynb` de su repositorio (el que crearon en la Clase 7). Copien la celda, usen solo las secciones que aplican a su dataset y borren las otras.

In [ ]:
# ==========================================================
# AMPLIACIÓN 03-eda (Clase 8) — completar con SU proyecto
# ==========================================================
# Usar solo las secciones que aplican a su dataset; borrar las otras.

# guardar_figura ya está en su 03-eda (Clase 7). Si no, es esta (RAIZ = Path("..") desde notebooks/):
# def guardar_figura(fig, nombre):
#     (RAIZ / "figs").mkdir(exist_ok=True)
#     fig.savefig(RAIZ / "figs" / nombre, dpi=150, bbox_inches="tight")

# 5. Incertidumbre: tasas (o promedios) por grupo con n e intervalo
#    - tabla: grupo · n · valor · lo · hi ; gráfico de puntos con barras de error
#    - si hay muchos grupos de tamaños muy distintos: gráfico de embudo

# 6. Tiempo (si tiene fechas): serie, estacionalidad (una línea por año), anomalías (heatmap unidad × mes)
#    - ojo con ceros que son datos faltantes, y con meses/semanas ausentes (reindex)

# 7. Muchos grupos o territorio: paneles con ejes compartidos (col_wrap) o cartograma/mapa de TASAS
#    - unir por código como texto con cero; no mapear conteos

# 8. (Opcional) Dashboard exploratorio: una función panel(grupo) y, si quieren, un HTML de Plotly en figs/

# Bitácora: agregar al menos 3 filas nuevas (quedan 6 o más), con las mismas 5 columnas de la Clase 7.
BITACORA_NUEVAS = [
    {"hallazgo": "",           # frase con número, n e intervalo si compara grupos
     "figura": "",             # figs/xx.png
     "celda": "",              # "03-eda.ipynb, celda N"
     "pregunta": "",           # qué término de la pregunta de datos toca
     "decision_o_hipotesis": ""},
    # ... mínimo 3 filas hoy
]
# En su 03-eda.ipynb:  BITACORA_EDA += BITACORA_NUEVAS ; pd.DataFrame(BITACORA_EDA)
pd.DataFrame(BITACORA_NUEVAS)

Reglas nuevas para la bitácora (se suman a las de la Clase 7):

- Todo hallazgo que **compara grupos** lleva **n e intervalo** (en la figura o en el pie). Si un grupo tiene n < 30, mostrar los puntos, no solo la barra.
- Paneles de varios grupos: **`sharey=True`**, para que la comparación sea a la misma escala.
- Territorio: cartograma o mapa de **tasas o cambios**, nunca de conteos; el código de comuna se une como **texto con cero** (`"08101"`, no `8101`).
- El **dashboard es opcional** y **no reemplaza la bitácora**: la bitácora sigue siendo la fuente de la sección de EDA del Informe 3, con sus 5 columnas.

---
### Checklist de salida de la Clase 8

- [ ] Ejercicios 1, 2 y 3 pasan su verificación automática (el 4 puede quedar para la casa)
- [ ] `notebooks/03-eda.ipynb` ampliado con la sección de la Clase 8
- [ ] Bitácora con 6 o más filas, cada una con figura en `figs/` y su celda
- [ ] Al menos una fila con n e intervalo, o un patrón en el tiempo, o una segmentación por grupo o territorio
- [ ] Lista corta de variables candidatas para la Clase 9, con una línea por variable